In [1]:
import json
import numpy as np
import scipy.sparse as sp
import networkx as nx
import matplotlib.pyplot as plt

import time
import copy

from msro import msro as algo, get_mean_row_front_size

### 1 - Test with the matrix given in the 1999 paper

In [448]:
# m = np.array([[1,0,1,1,0,0],
#               [0,1,0,1,1,0],
#               [1,0,1,1,0,1],
#               [0,1,0,0,0,0],
#               [0,0,0,1,1,1],
#               [0,0,0,0,0,1]])
m = np.array([[1,0,0,1,0,1],
              [0,1,0,0,1,0],
              [0,0,1,0,0,1],
              [0,1,0,0,0,0],
              [0,0,0,0,1,1],
              [1,0,1,1,0,1]])

msro_m = algo(m, perm = "rows")
print(msro_m)

[[1 0 0 1 0 1]
 [1 0 1 1 0 1]
 [0 0 1 0 0 1]
 [0 0 0 0 1 1]
 [0 1 0 0 1 0]
 [0 1 0 0 0 0]]


### 2 - Test with a `150x150` matrix taken from a random 3-SAT problem with `alpha` = 1

In [450]:
N = 150
alpha = 1
M = int(alpha * N)
sample = 49
graph = json.load(open(f"N{N}/{sample}.json"))
clauses = graph["clauses"]
m = np.zeros(shape = (M,N), dtype = int)
for i, clause in enumerate(clauses):
    m[i][clause] = 1

msro_m = algo(m, perm = "columns")

initial_frontSize = get_mean_row_front_size(m)
optimized_frontSize = get_mean_row_front_size(msro_m)
difference = (initial_frontSize - optimized_frontSize) / initial_frontSize

print(f"Initial mean row front size : {initial_frontSize}")
print(f"Optimized mean row front size : {optimized_frontSize}")
print(f"Mean row front size lowered by {100 * difference} %")

Initial mean row front size : 74.70666666666666
Optimized mean row front size : 32.233333333333334
Mean row front size lowered by 56.853471354631445 %


### 3 - Tests for a random `mxn` matrix

In [457]:
m = 1000  # Number of rows
n = 1000  # Number of columns

# Generate an empty matrix filled with zeros
matrix = np.zeros((m, n), dtype=int)

# Add at least 3 ones per row
for i in range(m):
    row_indices = np.random.choice(n, 3, replace=False)
    matrix[i, row_indices] = 1

# Add at least 3 ones per column
for j in range(n):
    col_indices = np.random.choice(m, 3, replace=False)
    matrix[col_indices, j] = 1

In [458]:
msro_m = algo(matrix, perm="rows")

### 4 - Tests to use sparse matrices in the algorithm to optimize its speed

In [ ]:
# To get the list of edges for the row graph
m = np.array([[1,0,1,1,0,0],
              [0,1,0,1,1,0],
              [1,0,1,1,0,1],
              [0,1,0,0,0,0],
              [0,0,0,1,1,1],
              [0,0,0,0,0,1]])

m_sparse = sp.csr_matrix(m)

A = m_sparse @ m_sparse.T
A_upper = sp.triu(A, k = 1)
upper_indices = A_upper.nonzero()
edges_list = np.column_stack((upper_indices[0], upper_indices[1]))
print(A)
print(A.shape)
print(edges_list)

In [ ]:
def get_rcgain(m: sp.csr_matrix, row_id: int, nb_already_assembled_rows: int):
    row = matrix.getrow(row_id)  # Get the specified row as a sparse matrix
    matrix = sp.vstack([matrix[:row_id], matrix[row_id+1:]])  # Remove the row from the matrix
    matrix = sp.vstack([matrix[:nb_already_assembled_rows], row, matrix[nb_already_assembled_rows:]])  # Insert the row at the desired position

    s = 0
    newc = 0
    for col_id in range(matrix.shape[1]):
        column = matrix[:, col_id]
        not_in_front = column[nb_already_assembled_rows + 1:]
        if not_in_front.nnz == 0:  # Check if there are no non-zero elements in `not_in_front`
            s += 1
        in_front = column[:nb_already_assembled_rows + 1]
        if np.all(in_front[:-1].data == 0) and in_front[-1] == 1:
            newc += 1

    rgain = 1 - s
    cgain = newc - s
    rcgain = rgain + cgain

    return rcgain

### 5 - Tests for the optimization of the MSRO algorithm

In [74]:
def get_rowGraph_data(m: np.ndarray, showG: bool = False):
    edges_list = np.argwhere(np.triu(m @ m.T, k = 1))

    Gr = nx.Graph()
    Gr.add_nodes_from(range(m.shape[0]))
    Gr.add_edges_from(edges_list)
    if showG:
        nx.draw(Gr, with_labels=True)
        plt.show()
    print("Finding the diameter of the graph...")
    start1 = time.time()
    diameter = nx.algorithms.distance_measures.diameter(Gr)
    end1 = time.time()
    print(f"Diameter found in {end1 - start1} s!")
    print("Finding the nodes tha are separated by this distance...")
    start2 = time.time()
    diameter_nodes = nx.algorithms.distance_measures.periphery(Gr)
    end2 = time.time()
    print(f"Nodes found in {end2 - start2} s!")
    connected_pairs = []
    for n1 in diameter_nodes:
        for n2 in diameter_nodes:
            if (
                n1 < n2
                and nx.shortest_path_length(Gr, source=n1, target=n2) == diameter
            ):
                connected_pairs.append((n1, n2))
    random_id = np.random.randint(len(connected_pairs))
    s, e = connected_pairs[random_id]

    distances = nx.shortest_path_length(Gr, target = e) # dictionnary of distances from the node `e`
    neighbors = {}
    for node in Gr.nodes():
        neighbors_i = nx.neighbors(Gr, node)
        neighbors[node] = list(neighbors_i)

    return s, e, distances, neighbors


def update_rows_order(rows_states: list,
                      row_to_assemble_id: int, # the current row id, not the original one
                      assembling_step: int,
                      neighbors: dict):
    rows_states_copy = copy.deepcopy(rows_states)
    original_row_id = rows_states_copy[row_to_assemble_id][0]
    rows_states_copy[assembling_step][0] = original_row_id
    for i in range(assembling_step+1, row_to_assemble_id+1):
        rows_states_copy[i][0] = rows_states[i - 1][0]
    # print(rows_states_copy)

    # Update the rows states (inactive, active or assembled)
    rows_states_copy[assembling_step][1] = "assembled"
    for neighbor in neighbors[original_row_id]:
        current_row_id = [k for k, v in rows_states_copy.items() if v[0] == neighbor][0] # get the current row idx from the original row idx
        if rows_states_copy[current_row_id][1] != "assembled":
            rows_states_copy[current_row_id][1] = 'active'

    return rows_states_copy


def get_rcgain_and_nold(m: np.ndarray,
                        rows_states: dict,
                        row_id: int, # the current row id
                        nb_assembled_rows: int,
                        neighbors: dict):
    # Switch the rows order in the matrix
    # m_copy = np.array(m, copy=True)
    rows_states = update_rows_order(rows_states,
                                    row_id,
                                    nb_assembled_rows,
                                    neighbors)
    new_rows_order = [i[0] for i in rows_states.values()]
    m_modified = m[new_rows_order]

    # Find the values of `s` (called `nold` here) and `newc`
    not_in_front_counts = np.sum(m_modified[nb_assembled_rows + 1:], axis = 0)
    nold = m_modified.shape[1] - np.count_nonzero(not_in_front_counts)
    in_front_counts = np.sum(matrix[:nb_assembled_rows + 1], axis=0)
    newc = np.count_nonzero((matrix[nb_assembled_rows] == 1) & (in_front_counts == 1))
    rgain = 1 - nold
    cgain = newc - nold
    rcgain = rgain + cgain
    return rcgain, nold


def initialize_P(m: np.ndarray,
                 rows_states: dict,
                 neighbors: dict,
                 distances: dict):
    W1 = 2
    W2 = 1
    P = []
    for i in range(m.shape[0]): # i must be the current row id
        rcgain_i, nold_i = get_rcgain_and_nold(m, rows_states, i, 0, neighbors)
        P_i = -W1 * rcgain_i + W2 * distances[i]
        P.append(P_i)
    return P


def get_row_to_assemble(P: list,
                        rows_states: dict,
                        s: int,
                        assembling_step: int,
                        smallest_value: int = -1e12):
    if assembling_step == 0:
        current_row_to_assemble = s
    else:
        not_active_rows_idx = [data[0] for data in rows_states.values() if data[1] != "active"]
        P_copy = np.array(P, copy = True)
        P_copy[not_active_rows_idx] = smallest_value
        original_row_to_assemble = np.argmax(P_copy)
        current_row_to_assemble = [k for k, v in rows_states.items() if v[0] == original_row_to_assemble][0]
    return current_row_to_assemble


def update_P(m: np.ndarray,
             P: list,
             rows_states: dict,
             neighbors: dict,
             distances: dict,
             smallest_value = -1e12):
    original_assembled_rows_idx = [v[0] for v in rows_states.values() if v[1] == "assembled"] # original assembled row ids
    nb_assembled_rows = len(original_assembled_rows_idx)
    last_assembled_row_idx = original_assembled_rows_idx[-1]
    P[last_assembled_row_idx] = smallest_value
    active_rows_idx = [k for k, v in rows_states.items() if v[1] == "active"] # current active row ids
    W1 = 2
    W2 = 1
    for i in active_rows_idx: # i must be the current row id
        original_i = [v[0] for k, v in rows_states.items() if k == i][0]
        # print(f"Current index : {i}, Original index : {original_i}")
        rcgain_i, nold_i = get_rcgain_and_nold(m, rows_states, i, nb_assembled_rows, neighbors)
        P_i = -W1 * rcgain_i + W2 * distances[original_i]
        P[original_i] = P_i
    return P


def msro(m: np.ndarray, perm: str = "columns", verbose: bool = False):
    if perm == "columns":
        matrix = np.array(m, copy=True).T
    elif perm == "rows":
        matrix = np.array(m, copy=True)
    else:
        raise f"Bad input for `perm`, please use 'rows' or 'columns'"

    rows_states = {}
    for i in range(matrix.shape[0]):
        rows_states[i] = [i, "inactive"]

    start_Gr = time.time()
    s, e, distances, neighbors = get_rowGraph_data(matrix)
    end_Gr = time.time()
    print(f"Done with finding data on the graph in {end_Gr - start_Gr} s...")

    for assembling_step in range(matrix.shape[0]):
        start = time.time()
        if assembling_step == 0:
            P = initialize_P(matrix, rows_states, neighbors, distances)
            row_id = get_row_to_assemble(P, rows_states, s, assembling_step)
            rows_states = update_rows_order(rows_states,
                                            row_id,
                                            assembling_step,
                                            neighbors)
            if verbose:
                print(f"Initial P : {P}")
                print(f"Next row to assemble : {row_id}")
                print(f"Next rows order : {rows_states}")
        else:
            P = update_P(matrix, P, rows_states, neighbors, distances)
            row_id = get_row_to_assemble(P, rows_states, s, assembling_step)
            rows_states = update_rows_order(rows_states,
                                            row_id,
                                            assembling_step,
                                            neighbors)
            if verbose:
                print(f"Updated P : {P}")
                print(f"Next row to assemble : {row_id}")
                print(f"Next rows order : {rows_states}")
        end = time.time()
        print(f"Row assembled after {end - start} seconds")
        nb_active_rows = len([v[0] for v in rows_states.values() if v[1] == "active"])
        print(f"Number of active rows : {nb_active_rows}")

    final_order = [v[0] for v in rows_states.values()]
    return final_order

In [87]:
value = 1
if value == 0:
    matrix = np.array([[1,0,1,1,0,0],
                       [0,1,0,1,1,0],
                       [1,0,1,1,0,1],
                       [0,1,0,0,0,0],
                       [0,0,0,1,1,1],
                       [0,0,0,0,0,1]])
elif value == 1:
    matrix = np.array([[1,0,0,1,0,1],
                       [0,1,0,0,1,0],
                       [0,0,1,0,0,1],
                       [0,1,0,0,0,0],
                       [0,0,0,0,1,1],
                       [1,0,1,1,0,1]])
elif value == 2:
    N = 150
    alpha = 1
    M = int(alpha * N)
    sample = 49
    graph = json.load(open(f"N{N}/{sample}.json"))
    clauses = graph["clauses"]
    matrix = np.zeros(shape = (M,N), dtype = int)
    for i, clause in enumerate(clauses):
        matrix[i][clause] = 1
elif value == 3:
    m = 1000  # Number of rows
    n = 1000  # Number of columns
    matrix = np.zeros((m, n), dtype=int)
    for i in range(m):
        row_indices = np.random.choice(n, 3, replace=False)
        matrix[i, row_indices] = 1
    for j in range(n):
        col_indices = np.random.choice(m, 2, replace=False)
        matrix[col_indices, j] = 1

perm = "rows"
new_order = msro(matrix, perm=perm, verbose=False)
# if perm == "rows":
#     print(m[new_order])
# elif perm == "columns":
#     print(m[:, new_order])
print(new_order)


Finding the diameter of the graph...
Diameter found in 0.00010561943054199219 s!
Finding the nodes tha are separated by this distance...
Nodes found in 6.103515625e-05 s!
Done with finding data on the graph in 0.0006682872772216797 s...
Row assembled after 0.0015299320220947266 seconds
Number of active rows : 3
Row assembled after 0.0004892349243164062 seconds
Number of active rows : 3
Row assembled after 0.0004982948303222656 seconds
Number of active rows : 2
Row assembled after 0.0003376007080078125 seconds
Number of active rows : 2
Row assembled after 0.0003237724304199219 seconds
Number of active rows : 1
Row assembled after 0.00021767616271972656 seconds
Number of active rows : 0
[2, 5, 0, 4, 1, 3]


In [77]:
matrix = np.array([[1,0,1,1,0,0],
                   [0,1,0,1,1,0],
                   [1,0,1,1,0,1],
                   [0,1,0,0,0,0],
                   [0,0,0,1,1,1],
                   [0,0,0,0,0,1]])
assembling_step = 0
for i in range(assembling_step, matrix.shape[1]):
    row = matrix[i]
    
    mask = (row == 1) & (matrix[:assembling_step+1, :].sum(axis=0) == 0)

In [12]:
arr = [[0,'0'],[3,'1'],[6,'2'],[9,"3"]]
dic = {}
np.where(arr[:,0] == 0)

TypeError: list indices must be integers or slices, not tuple

In [73]:
matrix = np.array([[1,0,1,1,0,0],
                   [0,1,0,1,1,0],
                   [1,0,1,1,0,1],
                   [0,1,0,0,0,0],
                   [0,0,0,1,1,1],
                   [0,0,0,0,0,1]])
nb_assembled_rows = 1
in_front_counts = np.sum(matrix[:nb_assembled_rows + 1], axis=0)
newc = np.count_nonzero((matrix[nb_assembled_rows] == 1) & (in_front_counts == 1))
# nb = matrix.shape[1] - np.count_nonzero(np.sum(matrix[nb_assembled_rows + 1:], axis=0))

[1 1 1 2 1 0]
2
